*Kaggle notebook link placeholder*

# Training and Fine-Tuning RoBERTa for Classification
## Classifying ANLI Premise-Hypothesis Pairs

**Author**

| Roll Number | Name |
|---|---|
| G25AIT2028 | Chaurasia Kamalkumar Lallanprasad |
| G25AIT2106 | Solanki Bhavik Pravinbhai |
| G25AIT2035 | Govardhan Kumar |
| G25AIT2057 | Mahesh Om Prakash Bali |

**Program:** PGD Artificial Intelligence — IIT Jodhpur  
**Assignment:** MLOps Assignment 2

---

This notebook demonstrates how to train and fine-tune a RoBERTa model for classification with the Hugging Face `transformers` library.

We fine-tune RoBERTa on the [ANLI dataset](https://huggingface.co/datasets/facebook/anli), where each example contains a `premise` and a `hypothesis`. The model predicts one of three Natural Language Inference labels:
- entailment
- neutral
- contradiction

**Basic steps involved in using RoBERTa and Hugging Face:**
1. Prepare train and test splits.
2. Encode text pairs into a format RoBERTa understands.
3. Combine encoded inputs and labels into dataset objects.
4. Load the pre-trained RoBERTa model.
5. Fine-tune the model using training data.
6. Predict new labels and evaluate performance on test data.


## Task 1: Notebook Setup — Download & Import into Kaggle

## Import necessary Python libraries and modules


First, we import the required Python libraries and modules. These include tools for dataset handling (`datasets`), model training (`transformers`, `torch`), baseline modeling (`sklearn`), and experiment tracking (`wandb`).

In [1]:
# !pip3 install -q -U transformers>=4.40.0 torch>=2.2.0 accelerate>=1.1.0 datasets sentencepiece nbformat wandb gdown scikit-learn pandas numpy requests matplotlib seaborn ipywidgets huggingface_hub certifi python-dotenv

import os, certifi
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()

# os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")
# os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# Remove any stale wandb IPython hooks left over from a previous session.
_ip = get_ipython()
if _ip is not None:
    for _event in ('pre_run_cell', 'post_run_cell'):
        _stale = [
            cb for cb in list(_ip.events.callbacks.get(_event, []))
            if 'wandb' in str(type(getattr(cb, '__self__', cb))).lower()
        ]
        for cb in _stale:
            try:
                _ip.events.unregister(_event, cb)
            except Exception:
                pass

try:
    import wandb
    wandb.teardown()
except Exception:
    pass


In [2]:
# Basic Python modules
from collections import defaultdict
import random
import pickle

# For downloading large files from Google Drive
# https://github.com/wkentaro/gdown
import gdown

# For working with gzip files
# https://docs.python.org/3/library/gzip.html
import gzip

# For working with JSON files
import json

# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# For deep learning
# https://pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html
import torch

# For plotting and data visualization
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import ticker
sns.set(style='ticks', font_scale=1.2)

from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
 

ModuleNotFoundError: No module named 'gdown'

In [ ]:
# Run-mode key (single place to switch execution scale)
RUN_MODE = 'SMALL_RUN'  # Options: 'SMALL_RUN' or 'FULL_RUN'

if RUN_MODE not in {'SMALL_RUN', 'FULL_RUN'}:
    raise ValueError("RUN_MODE must be either 'SMALL_RUN' or 'FULL_RUN'.")

SMALL_RUN = RUN_MODE == 'SMALL_RUN'
FULL_RUN = RUN_MODE == 'FULL_RUN'

print(f'RUN_MODE selected: {RUN_MODE}')

The HuggingFace [`transformers` Python library](https://huggingface.co/transformers/installation.html) is included in Colab by default now, so we do not need to install it (but this is how you would install it with `pip`).

From `transformers`, we will import modules for `RoBERTa`, a **R**obustly **O**ptimized **BERT** **P**re-training **A**pproach developed by Facebook AI Research. RoBERTa improves upon the original BERT by removing the Next Sentence Prediction (NSP) objective (shown to be sub-optimal), training with **dynamic masking** (a fresh random mask per epoch rather than a fixed one at preprocessing time), using much larger mini-batches, and training on significantly more data for longer. The `roberta-base` variant comes with 12 transformer layers, a hidden size of 768, 12 attention heads, ~125M parameters, and a **byte-level BPE** vocabulary of 50,000 tokens — trained on 160GB of diverse English text.


In [ ]:
#from transformers import DistilBertForSequenceClassification , DistilBertTokenizerFast
#from transformers import DebertaV2ForSequenceClassification, DebertaV2Tokenizer
from transformers import RobertaForSequenceClassification , RobertaTokenizer
from transformers import Trainer, TrainingArguments

## Set parameters and file paths


In [ ]:
# HF_MODEL_CLASS = DistilBertForSequenceClassification
# HF_TOKENIZER_CLASS = DistilBertTokenizerFast
# model_name = 'distilbert-base-cased'

# HF_MODEL_CLASS = DebertaV2ForSequenceClassification
# HF_TOKENIZER_CLASS = DebertaV2Tokenizer
# model_name = 'microsoft/deberta-v3-small'

HF_MODEL_CLASS = RobertaForSequenceClassification
HF_TOKENIZER_CLASS = RobertaTokenizer
model_name = 'roberta-base'

# Device: 'mps' for Apple Silicon, 'cuda' for NVIDIA GPU, 'cpu' as fallback
device_name = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device_name}")

# Maximum number of tokens per input sequence
max_length = 512

# Directory to save the fine-tuned model
cached_model_directory_name = 'anli-classification-roberta'

# Full run defaults
DATA_SET_HEAD = 10000
DATA_SET_SAMPLE = 2000
TRAIN_SPLITS = ['train_r1', 'train_r2', 'train_r3']
TEST_SPLITS = ['test_r1', 'test_r2', 'test_r3']

# Tiny run overrides
if SMALL_RUN:
    DATA_SET_HEAD = 300
    DATA_SET_SAMPLE = 100
    TRAIN_SPLITS = ['train_r1']
    TEST_SPLITS = ['test_r1']

print(f"SMALL_RUN: {SMALL_RUN}")
print(f"FULL_RUN: {FULL_RUN}")
print(f"Train splits: {TRAIN_SPLITS}, rows/split: {DATA_SET_HEAD}")
print(f"Test splits: {TEST_SPLITS}, rows/split: {DATA_SET_SAMPLE}")


## Load and prepare ANLI data


In this section, we load the ANLI dataset directly from Hugging Face Datasets using `load_dataset("facebook/anli")`.

We build text inputs by concatenating `premise` and `hypothesis`, and use ANLI labels (`entailment`, `neutral`, `contradiction`) as class names.

In [ ]:
from datasets import load_dataset

# Load ANLI from Hugging Face Datasets
ds = load_dataset("facebook/anli")

print(ds)

Next, we convert ANLI examples into text-classification inputs by joining each `premise` and `hypothesis` into one string, then build train/test text-label lists for model training.

In [ ]:
from collections import Counter

label_names = ds['train_r1'].features['label'].names
label_id_to_name = {i: name for i, name in enumerate(label_names)}

def build_text(example):
    return f"premise: {example['premise']} hypothesis: {example['hypothesis']}"

def collect_split(split_name, max_rows=None):
    split = ds[split_name].shuffle(seed=42)
    if max_rows is not None:
        split = split.select(range(min(max_rows, len(split))))

    texts = [build_text(ex) for ex in split]
    labels = [label_id_to_name[int(ex['label'])] for ex in split]
    return texts, labels

# Build train set from selected ANLI train rounds
all_train_texts, all_train_labels = [], []
for split_name in TRAIN_SPLITS:
    _texts, _labels = collect_split(split_name, max_rows=DATA_SET_HEAD)
    all_train_texts.extend(_texts)
    all_train_labels.extend(_labels)

# Build test set from selected ANLI test rounds
all_test_texts, all_test_labels = [], []
for split_name in TEST_SPLITS:
    _texts, _labels = collect_split(split_name, max_rows=DATA_SET_SAMPLE)
    all_test_texts.extend(_texts)
    all_test_labels.extend(_labels)

print(f"Train examples: {len(all_train_texts)}")
print(f"Test examples: {len(all_test_texts)}")
print('Train label distribution:', Counter(all_train_labels))
print('Test label distribution:', Counter(all_test_labels))

Let's preview a couple of ANLI text-label examples from the prepared training split.

In [ ]:
for _text, _label in random.sample(list(zip(all_train_texts, all_train_labels)), 2):
   print("LABEL:", _label)
   print("TEXT:", _text[:300], "...")
   print()

Here we use `pickle` to save this Python dictionary to a `.pickle` file so we can easily load it later.

*The `pickle` module allows you to save and load Python objects like lists and dictionaries.*

In [ ]:
pickle.dump(
    {
        'train_texts': all_train_texts,
        'train_labels': all_train_labels,
        'test_texts': all_test_texts,
        'test_labels': all_test_labels,
    },
    open('anli_text_classification_data.pickle', 'wb')
)

## Prepare training and test sets


For ANLI, we already prepared train and test splits from the official dataset rounds (`train_r1/r2/r3` and `test_r1/r2/r3`).

In this step, we assign those prepared splits to the variables used by the rest of the notebook.

In [ ]:
train_texts = list(all_train_texts)
train_labels = list(all_train_labels)

test_texts = list(all_test_texts)
test_labels = list(all_test_labels)

Show how many ANLI texts and labels we have in each split.

In [ ]:
len(train_texts), len(train_labels), len(test_texts), len(test_labels)

Here's an example of a training label and review:

In [ ]:
train_labels[0], train_texts[0]

## Run a baseline model (logistic regression)

Here we train and evaluate a simple TF-IDF baseline model using logistic regression on ANLI text pairs.

This gives a quick reference point before fine-tuning RoBERTa.


In [ ]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

We train a logistic regression model from scikit-learn on the ANLI training split, and then use it to predict labels for the ANLI test split.

In [ ]:
model = LogisticRegression(max_iter=1000).fit(X_train, train_labels)
predictions = model.predict(X_test)

We use scikit-learn's `classification_report` to evaluate how well predictions match the true ANLI labels.

For ANLI, random performance is around 0.33 because there are 3 classes.

In [ ]:
print(classification_report(test_labels, predictions))

## Encode data for RoBERTa

We transform our ANLI inputs and labels into a format RoBERTa (via Hugging Face and PyTorch) understands.

Here are the steps:

1. Convert textual labels into integer IDs.

2. Tokenize paired text input (`premise` + `hypothesis`) with truncation and padding to `max_length`.

3. Add RoBERTa special tokens during tokenization.

| Special token | Explanation |
|---|---|
| `<s>` | Start token of sequence |
| `</s>` | End/separator token |
| `<pad>` | Padding token |
| `Ġ` | Byte-level BPE word-start prefix |


## Task 2: Load Pre-trained Model from Hugging Face

### Model Selection Rationale

For this NLI classification task, `FacebookAI/roberta-base` is used because it is robust, widely validated, and integrates cleanly with the Hugging Face `Trainer` workflow.

RoBERTa removes the Next Sentence Prediction objective, uses dynamic masking, and is trained on larger and more diverse corpora than the original BERT setup. These characteristics make it a strong baseline for sentence-pair inference tasks such as ANLI.

| Dimension | `distilbert-base-cased` | `FacebookAI/roberta-base` ✅ |
|-----------|------------------------|------------------------------|
| Attention mechanism | Standard self-attention | Standard self-attention |
| Pre-training objective | Distilled BERT objective | Masked LM with dynamic masking, no NSP |
| Tokenizer | WordPiece | Byte-level BPE (50K vocab) |
| Parameters | ~66M | ~125M |
| Typical tradeoff | Faster, smaller | Better capacity, stronger baseline |


### Step 1 — Load the Tokenizer from Hugging Face Hub

We load `RobertaTokenizer` from the Hugging Face Hub using `from_pretrained(model_name)`. RoBERTa uses a byte-level BPE tokenizer, where word-start tokens are prefixed with `Ġ`. The tokenizer truncates each ANLI text pair to 512 tokens and adds special tokens `<s>` (start), `</s>` (separator/end), and `<pad>` (padding).


In [ ]:
tokenizer = HF_TOKENIZER_CLASS.from_pretrained(model_name) # The model_name needs to match our pre-trained model.

Here we create a mapping from ANLI label names to integer IDs. We take the unique labels and build dictionaries for both directions (`label2id` and `id2label`).

**Note:** Hugging Face documentation may use both "labels" and "tags"; in this notebook we consistently use "labels."

In [ ]:
# Keep label mapping consistent with ANLI dataset definition:
# entailment (0), neutral (1), contradiction (2)
label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for label, idx in label2id.items()}

In [ ]:
label2id.keys()

In [ ]:
id2label.keys()

Now let's encode our texts and labels!

In [ ]:
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=max_length)
test_encodings  = tokenizer(test_texts, truncation=True, padding=True, max_length=max_length)

train_labels_encoded = [label2id[y] for y in train_labels]
test_labels_encoded  = [label2id[y] for y in test_labels]

**Examine a training example after tokenization**

In [ ]:
' '.join(tokenizer.convert_ids_to_tokens(train_encodings['input_ids'][0][:100]))

**Examine a test example after tokenization**

In [ ]:
' '.join(tokenizer.convert_ids_to_tokens(test_encodings['input_ids'][0][:100]))

**Examine the training labels after encoding**

In [ ]:
set(train_labels_encoded)

**Examine the test labels after encoding**

In [ ]:
set(test_labels_encoded)

## Make a custom Torch dataset


Here we combine the encoded labels and texts into dataset objects. We use the custom Torch `MyDataSet` class to make a `train_dataset` object from  the `train_encodings` and `train_labels_encoded`. We also make a `test_dataset` object from `test_encodings`, and `test_labels_encoded`.

In [ ]:
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = MyDataset(train_encodings, train_labels_encoded)
test_dataset = MyDataset(test_encodings, test_labels_encoded)

**Examine a tokenized input from `train_dataset`**

In [ ]:
' '.join(tokenizer.convert_ids_to_tokens(train_dataset.encodings['input_ids'][0][:100]))

**Examine a tokenized input from `test_dataset`**

In [ ]:
' '.join(tokenizer.convert_ids_to_tokens(test_dataset.encodings['input_ids'][1][:100]))

## Load pre-trained RoBERTa model


### Step 2 — Load the Pre-trained Model from Hugging Face Hub

We load `RobertaForSequenceClassification` with `num_labels=len(id2label)` so the classification head matches the ANLI label set (`entailment`, `neutral`, `contradiction`).

The base transformer weights come from the pre-trained `FacebookAI/roberta-base` checkpoint on Hugging Face Hub. Only the randomly initialized classification head is trained from scratch during fine-tuning.

> **Note:** Re-run this cell if you repeat fine-tuning, since it resets model weights to the original pre-trained checkpoint.


In [ ]:
# Task 2 — Load pre-trained RoBERTa from Hugging Face Hub
# model_name must match the tokenizer loaded above.
# num_labels = number of unique ANLI labels so the classification head has the right output size.
model = HF_MODEL_CLASS.from_pretrained(
    model_name,
    num_labels=len(id2label),
).to(device_name)

print(f"Model loaded: {model_name}")
print(f"Number of output labels: {len(id2label)}  →  {list(id2label.values())}")
print(f"Device: {device_name}")


## Set the RoBERTa fine-tuning parameters

These are the arguments we'll set in the HuggingFace `TrainingArguments` object. The most important are the **number of training epochs** and the **learning rate**.

When training your own model, you should search over these parameters to find the best values for your specific use case.


| Parameter | Explanation |
|-----------| ------------|
| num_train_epochs | total number of training epochs (how many times to pass through the entire dataset; too much can cause overfitting) |
| per_device_train_batch_size | batch size per device during training |
| per_device_eval_batch_size |  batch size for evaluation |
|  warmup_steps |  number of warmup steps for learning rate scheduler (set lower because of small dataset size) |
| weight_decay | strength of weight decay (reduces size of weights, like regularization) |
| output_dir | output directory for the fine-tuned model and configuration files |
| logging_dir | directory for storing logs |
| logging_steps | how often to print logging output (so that we can stop training early if the loss isn't going down) |
| evaluation_strategy | evaluate while training so that we can see the accuracy going up |

In [ ]:
# Distilbert

# training_args = TrainingArguments(
#     num_train_epochs=3,
#     per_device_train_batch_size=10,
#     per_device_eval_batch_size=16,
#     learning_rate=5e-5,
#     warmup_steps=100,
#     weight_decay=0.01,
#     output_dir='./results',
#     logging_dir='./logs',
#     logging_steps=50,
#     eval_strategy='steps',
#     eval_steps=500,
#     save_strategy='steps',
#     save_steps=500,
#     load_best_model_at_end=True,
#     report_to='wandb',          # Task 4: enables full W&B experiment tracking
#     run_name=f'{model_name}-run-{timestamp}',
# )

# Deberta-v3-small

# training_args = TrainingArguments(
#     output_dir='./results',
#     num_train_epochs=3,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=16,
#     learning_rate=2e-5,                   # Bug 2 fix
#     warmup_steps=100,
#     weight_decay=0.01,
#     logging_steps=50,
#     eval_strategy='epoch',
#     save_strategy='epoch',
#     load_best_model_at_end=True,
#     metric_for_best_model='eval_accuracy', # Bug 4 fix
#     greater_is_better=True,               # Bug 4 fix
#     fp16=False,                           # Bug 1 fix
#     bf16=False,                           # Bug 1 fix
#     report_to='wandb',
#     run_name=f'{model_name}-run-{timestamp}',
# )

if SMALL_RUN:
    training_args = TrainingArguments(
        num_train_epochs=1,
        max_steps=80,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        warmup_steps=0,
        weight_decay=0.01,
        output_dir='./results',
        logging_dir='./logs',
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=20,
        save_strategy='no',
        load_best_model_at_end=False,
        report_to='none',
        run_name=f'{model_name}-smoke-{timestamp}',
    )
else:
    training_args = TrainingArguments(
        num_train_epochs=3,
        per_device_train_batch_size=10,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        warmup_steps=100,
        weight_decay=0.01,
        output_dir='./results',
        logging_dir='./logs',
        logging_steps=50,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        report_to='wandb',
        run_name=f'{model_name}-run-{timestamp}',
    )

print('Training mode:', 'SMALL_RUN' if SMALL_RUN else 'FULL_RUN')

## Fine-tune the RoBERTa model


## Task 3: Train the Model & Track with W&B

This section fine-tunes `FacebookAI/roberta-base` using the Hugging Face `Trainer` API and logs metrics, hyperparameters, and training curves to Weights & Biases via `report_to="wandb"`.

**What W&B captures automatically:**

| What W&B logs | Why it matters |
|---|---|
| Training loss | Shows if the model is learning |
| Validation loss | Signals overfitting or underfitting |
| Accuracy & F1 | Core NLI evaluation metrics |
| Learning rate schedule | Tracks optimizer behavior |
| Hyperparameters | Ensures reproducibility |


First, we define a custom evaluation function that returns the accuracy. You could modify this function to return precision, recall, F1, and/or other metrics.

In [ ]:
from sklearn.metrics import f1_score

# Task 4 — compute_metrics returns both accuracy and weighted F1
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
    }

Then we create a HuggingFace `Trainer` object using the `TrainingArguments` object that we created above. We also send our `compute_metrics` function to the `Trainer` object, along with our test and train datasets.

**Note:** This is what we've been aiming for this whole time! All the work of tokenizing, creating datasets, and setting the training arguments was for this cell.

In [ ]:
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=test_dataset,           # evaluation dataset (usually a validation set; here we just send our test set)
    compute_metrics=compute_metrics      # our custom evaluation function
)

Time to fine-tune.

If you run on GPU, this is usually much faster than CPU.

At each logging/evaluation interval, the trainer reports training loss, validation loss, and evaluation metrics from `compute_metrics`.

You should generally see loss decreasing and metrics improving over epochs; if not, tune learning rate, batch size, and number of epochs.

In [ ]:
if not SMALL_RUN:
    import wandb

    # Task 3 — Initialise W&B run and log all hyperparameters
    train_run = wandb.init(
        project='MLOps-ANLI-NLI',
        entity='kamalchaurasia-iit-jodhpur',
        name=f'{model_name}-run-{timestamp}',
        config={
            'model':           model_name,
            'epochs':          int(training_args.num_train_epochs),
            'batch_size':      training_args.per_device_train_batch_size,
            'learning_rate':   training_args.learning_rate,
            'max_length':      max_length,
            'dataset':         'facebook/anli',
            'num_labels':      len(id2label),
            'platform':        device_name,
        },
    )
    print('W&B run initialised. View at:', wandb.run.url)
else:
    print('SMALL_RUN enabled: skipping W&B initialisation.')

In [ ]:
trainer.train()


## Save fine-tuned model

The following cell will save the model and its configuration files to a local directory. To preserve this model for future use, back it up to a safe location or push it to the HuggingFace Hub.


In [ ]:
#trainer.save_model(cached_model_directory_name)

(Optional) If you've already fine-tuned and saved the model, you can reload it using the following line. You don't have to run fine-tuning every time you want to evaluate.

In [ ]:
# To reload the saved DeBERTa model without re-training:
# model = DebertaV2ForSequenceClassification.from_pretrained(cached_model_directory_name)
# model = DebertaV2ForSequenceClassification.from_pretrained(cached_model_directory_name)


## Task 4: Evaluate the Model & Save Results

After training, we evaluate the fine-tuned RoBERTa model on the held-out ANLI test split.

## Evaluate fine-tuned model

The following `Trainer.evaluate()` call runs the built-in evaluation loop, computing loss, accuracy, and F1 via our `compute_metrics` function.


In [ ]:
# Task 4 — Step 1: run built-in evaluation (loss + accuracy + F1)
eval_results = trainer.evaluate()
print(eval_results)

# Task 4 — Step 2: log final metrics to W&B explicitly (full run only)
if not SMALL_RUN:
    import wandb
    wandb.log({
        'final/loss':     eval_results['eval_loss'],
        'final/accuracy': eval_results['eval_accuracy'],
        'final/f1':       eval_results['eval_f1'],
    })
else:
    print('SMALL_RUN enabled: skipping W&B metric logging.')


But we might want to do more fine-grained analysis of the model, so we extract the predicted labels.

In [ ]:
predicted_results = trainer.predict(test_dataset)

In [ ]:
predicted_results.predictions.shape

In [ ]:
predicted_labels = predicted_results.predictions.argmax(-1) # Get the highest probability prediction
predicted_labels = predicted_labels.flatten().tolist()      # Flatten the predictions into a 1D list
predicted_labels = [id2label[l] for l in predicted_labels]  # Convert from integers back to strings for readability

In [ ]:
len(predicted_labels)

In [ ]:
print(classification_report(test_labels,
                            predicted_labels))

In [ ]:
import json
from sklearn.metrics import classification_report as skl_classification_report

# Task 4 — Step 3: save evaluation report to eval_report.json
# Use integer-encoded labels so target_names ordering aligns with id2label
preds  = predicted_results.predictions.argmax(-1)
labels = [item['labels'].item() for item in test_dataset]
report = skl_classification_report(
    labels,
    preds,
    target_names=list(id2label.values()),
    output_dict=True,
)
report['eval_loss']     = eval_results.get('eval_loss')
report['eval_accuracy'] = eval_results.get('eval_accuracy')
report['eval_f1']       = eval_results.get('eval_f1')

eval_report_path = 'eval_report.json'
with open(eval_report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Saved evaluation report → {eval_report_path}')
print(f"  accuracy : {report['eval_accuracy']:.4f}")
print(f"  f1       : {report['eval_f1']:.4f}")
print(f"  loss     : {report['eval_loss']:.4f}")

In [ ]:
if not SMALL_RUN:
    import wandb

    # Task 4 — Step 4: upload eval_report.json as a versioned W&B Artifact
    artifact = wandb.Artifact('eval-report', type='evaluation')
    artifact.add_file(eval_report_path)
    wandb.log_artifact(artifact)
    print(f'Uploaded "{eval_report_path}" as W&B Artifact "eval-report".')
    print(f'W&B run: {wandb.run.url}')
else:
    print('SMALL_RUN enabled: skipping W&B artifact upload.')

## Pull out correct and incorrect classifications for examination

Let's use the predicted labels for analysis.

Now that fine-tuning and prediction are complete, the remaining steps use standard Python tools (`pandas`, `scikit-learn`) to inspect errors and confusion patterns.

First, we print some correctly classified examples.


In [ ]:
for _true_label, _predicted_label, _text in random.sample(list(zip(test_labels, predicted_labels, test_texts)), 20):
  if _true_label == _predicted_label:
    print('LABEL:', _true_label)
    print('TEXT PAIR:', _text[:160], '...')
    print()

Now let's print out some misclassifications.

In [ ]:
for _true_label, _predicted_label, _text in random.sample(list(zip(test_labels, predicted_labels, test_texts)), 20):
  if _true_label != _predicted_label:
    print('TRUE LABEL:', _true_label)
    print('PREDICTED LABEL:', _predicted_label)
    print('TEXT PAIR:', _text[:160], '...')
    print()

Finally, let's create heatmaps to examine misclassification patterns across ANLI labels.

In [ ]:
label_classifications_dict = defaultdict(int)
for _true_label, _predicted_label in zip(test_labels, predicted_labels):
  label_classifications_dict[(_true_label, _predicted_label)] += 1

dicts_to_plot = []
for (_true_label, _predicted_label), _count in label_classifications_dict.items():
  dicts_to_plot.append({'True Label': _true_label,
                        'Predicted Label': _predicted_label,
                        'Number of Classifications': _count})

df_to_plot = pd.DataFrame(dicts_to_plot)
df_wide = df_to_plot.pivot_table(index='True Label',
                                 columns='Predicted Label',
                                 values='Number of Classifications')

In [ ]:
plt.figure(figsize=(9,7))
sns.set(style='ticks', font_scale=1.2)
sns.heatmap(df_wide, linewidths=1, cmap='Purples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Looks good. We can see where the model is assigning correct labels consistently.

Now, let's remove the diagonal from the plot to highlight only misclassifications.

In [ ]:
label_classifications_dict = defaultdict(int)
for _true_label, _predicted_label in zip(test_labels, predicted_labels):
  if _true_label != _predicted_label: # Remove the diagonal to highlight misclassifications
    label_classifications_dict[(_true_label, _predicted_label)] += 1

dicts_to_plot = []
for (_true_label, _predicted_label), _count in label_classifications_dict.items():
  dicts_to_plot.append({'True Label': _true_label,
                        'Predicted Label': _predicted_label,
                        'Number of Classifications': _count})

df_to_plot = pd.DataFrame(dicts_to_plot)
df_wide = df_to_plot.pivot_table(index='True Label',
                                 columns='Predicted Label',
                                 values='Number of Classifications')

In [ ]:
plt.figure(figsize=(9,7))
sns.set(style='ticks', font_scale=1.2)
sns.heatmap(df_wide, linewidths=1, cmap='Purples')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

There is much more you can do with your own text-pair datasets and labels. This workflow can be adapted to many NLI and sentence-pair classification tasks.

## Task 5: Push Trained Model to Hugging Face Hub

After fine-tuning, push the model and tokenizer to your public Hugging Face profile so it can be loaded by anyone.

In [ ]:
from huggingface_hub import login
import os

# Task 5 — Authenticate with Hugging Face (set HF_TOKEN env var or paste token)
# Never hardcode tokens in notebooks. Set the env var before running:
#   export HF_TOKEN="hf_..."
HF_TOKEN = os.environ.get('HF_TOKEN')
HF_REPO_ID = 'kamalchaurasia-iitj/mlops-anli-classifier-roberta'

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # prompts interactively

# Push model and tokenizer to your Hugging Face profile
# HF_REPO_ID = 'your-username/roberta-anli-classifier'
model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f'Model pushed to: https://huggingface.co/{HF_REPO_ID}')

# Record the HF model link in W&B summary only when a run is active
try:
    import wandb
    if wandb.run is not None:
        wandb.run.summary['huggingface_model'] = f'https://huggingface.co/{HF_REPO_ID}'
        wandb.finish()
    else:
        print('No active W&B run found. Skipping wandb.run.summary and wandb.finish().')
except Exception as e:
    print(f'W&B update skipped: {e}')


In [ ]:
from huggingface_hub import ModelCard

# Build model card content
labels = list(id2label.values())
label_list = "\n".join(f"- `{label}`" for label in labels)

# Pull key metrics from eval_results (populated by trainer.evaluate())
acc  = eval_results.get("eval_accuracy", eval_results.get("accuracy", "N/A"))
f1   = eval_results.get("eval_f1",       eval_results.get("f1", "N/A"))
loss = eval_results.get("eval_loss",     eval_results.get("loss", "N/A"))

if isinstance(acc,  float): acc  = f"{acc:.4f}"
if isinstance(f1,   float): f1   = f"{f1:.4f}"
if isinstance(loss, float): loss = f"{loss:.4f}"

card_content = f"""---
language: en
license: mit
tags:
  - text-classification
  - natural-language-inference
  - anli
  - roberta
  - fine-tuned
datasets:
  - facebook/anli
metrics:
  - accuracy
  - f1
model-index:
  - name: {HF_REPO_ID}
    results:
      - task:
          type: text-classification
          name: Natural Language Inference
        dataset:
          type: facebook/anli
          name: ANLI
        metrics:
          - type: accuracy
            value: {acc}
          - type: f1
            value: {f1}
---

# ANLI NLI Classifier (RoBERTa)

Fine-tuned [`{model_name}`](https://huggingface.co/{model_name}) for **3-class natural language inference**
on the [ANLI dataset](https://huggingface.co/datasets/facebook/anli).

## Labels

{label_list}

## Training details

| Parameter | Value |
|-----------|-------|
| Base model | `{model_name}` |
| Epochs | `{training_args.num_train_epochs}` |
| Learning rate | `{training_args.learning_rate}` |
| Train batch size | `{training_args.per_device_train_batch_size}` |
| Warmup steps | `{training_args.warmup_steps}` |
| Weight decay | `{training_args.weight_decay}` |

## Evaluation (test set)

| Metric | Score |
|--------|-------|
| Accuracy | {acc} |
| Weighted F1 | {f1} |
| Loss | {loss} |

## Usage

```python
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="{HF_REPO_ID}",
    tokenizer="{HF_REPO_ID}",
)

text = "premise: A man is playing guitar on stage. hypothesis: A person is performing music."
result = classifier(text)
print(result)
```

## Citation

```
@misc{{anli-nli-classifier,
  author = {{Kamalkumar Chaurasia}},
  title  = {{ANLI NLI Classifier ({model_name})}},
  year   = {{2026}},
  url    = {{https://huggingface.co/{HF_REPO_ID}}}
}}
```
"""

# Push model card to the Hub
card = ModelCard(card_content)
card.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
print(f"Model card updated: https://huggingface.co/{HF_REPO_ID}")
